# Answers: 8 · Tests and bugs

Worked answers to the **Try it yourself** exercises at the end of
[`8_tests_and_bugs.ipynb`](../8_tests_and_bugs.ipynb). Try each one yourself first. If you're stuck, run the cell here and
compare. Each answer runs on the same real data as the lesson, so you'll see real results,
not made-up ones.

The first code cell recreates just what the exercises need from the lesson, so this notebook
runs on its own.

In [ ]:
import sys; sys.path.insert(0, "..")
import io, contextlib, inspect
import numpy as np, pandas as pd, torch
from tour import *
import surrogates
sys.path.insert(0, str(AL_REPO / "tests"))
import test_rank_fusion as T

class MonkeyPatch:
    def __init__(self):
        self.saved = []
    def setattr(self, obj, name, value):
        self.saved.append((obj, name, getattr(obj, name)))
        setattr(obj, name, value)
    def undo(self):
        for obj, name, old in reversed(self.saved):
            setattr(obj, name, old)

def run_tests(module):
    results = {}
    for name, fn in vars(module).items():
        if not name.startswith("test_"):
            continue
        mp = MonkeyPatch()
        try:
            with contextlib.redirect_stdout(io.StringIO()):
                fn(mp) if "monkeypatch" in inspect.signature(fn).parameters else fn()
            results[name] = "PASS"
        except AssertionError:
            results[name] = "FAIL"
        finally:
            mp.undo()
    return pd.Series(results, name="result")
print("ready")

---
## Exercise 1: re-introduce bug 1 (`sigma` always zero)

In [ ]:
real_combine = surrogates.EnsembleFusionSurrogate.combine

def combine_with_zero_sigma(self, comps):
    mu, _ = real_combine(self, comps)             # keep the real ranking ...
    return mu, np.zeros(len(comps), dtype=np.float32)   # ... but throw the uncertainty away

mp = MonkeyPatch()
mp.setattr(surrogates.EnsembleFusionSurrogate, "combine", combine_with_zero_sigma)
try:
    print(run_tests(T))
finally:
    mp.undo()

**Answer.** Exactly one test fails: `test_ensemble_ucb_differs_from_greedy_and_matches_docstring`. It checks `assert sigma.max() > 0` ("there must be some uncertainty") and that UCB's batch differs from greedy's, and with zero sigma both are false. All the other tests still pass, because the *ranking* (`mu`) is unchanged. That's a good test suite: each test guards one specific property, so when one fails its name tells you what broke.

---
## Exercise 2: 10 chunks instead of 5

In [ ]:
scores, embeddings = load_mini()
y = -scores["score"].to_numpy(np.float32)
ALL = list(BACKBONES)
dims = {k: embeddings[k].shape[1] for k in ALL}
X = np.concatenate([embeddings[k] for k in ALL], axis=1)
train = np.random.default_rng(0).permutation(len(y))[:420]
torch.manual_seed(0)
fusion = surrogates.EnsembleFusionSurrogate(dims=dims)
with contextlib.redirect_stdout(io.StringIO()):
    fusion.fit(X[train], y[train])
comps_sorted = fusion.predict_components(X)[np.argsort(-y)]      # pool sorted best-first

for n_chunks in [1, 5, 10, 20]:
    size = len(y) // n_chunks + 1
    mu = np.concatenate([fusion.combine(comps_sorted[i:i + size])[0] for i in range(0, len(y), size)])
    batch = np.argsort(-mu)[:210]
    from_best_chunk = (batch < size).sum()
    print(f"{n_chunks:2d} chunks: best-100 in batch {(batch < 100).sum():3d}   "
          f"molecules taken from the best chunk {from_best_chunk:3d} of 210")

**Answer.** Yes, it gets steadily worse with more chunks. With one global ranking, all 210 picks come from the best chunk and the batch holds 37 of the true best 100. With 5 chunks only 61 picks come from the best chunk (20 of the best 100), with 10 chunks 32 picks (12), and with 20 chunks just 14 picks (9). More chunks means more fake "rank 1" molecules, one from every chunk, however bad that chunk is. On the full 2.1M pool, predicted in 50,000-molecule chunks, that's 43 chunks. No wonder fixing this was worth several points of recall.

---
## Exercise 3: a test for `random_recall`

In [ ]:
best100 = set(scores.nsmallest(100, "score")["smiles"])

def random_recall(n_docked, seed):                 # copied from notebook 2
    rng = np.random.default_rng(seed)
    picked = rng.choice(len(scores), n_docked, replace=False)
    return sum(s in best100 for s in scores["smiles"][picked]) / 100

def test_random_recall_is_about_six_percent():
    expected = 1260 / len(scores)                  # 0.0599: docking 6% finds 6% on average
    tries = [random_recall(1260, seed) for seed in range(50)]
    assert abs(np.mean(tries) - expected) < 0.01, f"mean {np.mean(tries):.3f}, expected {expected:.3f}"
    assert max(tries) < 0.2, "one random try found far too many, something is off"

test_random_recall_is_about_six_percent()
print("PASS")

**Answer.** One way to write it is shown above. Two design choices are worth copying. First, the test averages **many** tries (50 seeds) rather than trusting one, because a single random draw can easily land on 3% or 9%. Second, it allows a **tolerance** (within 0.01 of the expected 0.0599), because random results are never exact. A test that demanded exactly 6% would fail by chance and train you to ignore failing tests, which is the worst possible habit. The second `assert` is a sanity check: no single random try should find anywhere near 20% of the best 100.